## Setup

- Set `OPENAI_API_KEY`
- Upload to PDF in session storage

In [ ]:
!pip install -q chromadb openai pypdf

In [ ]:
import base64

from IPython.display import Audio, Image
from chromadb import PersistentClient
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from pathlib import Path
from pydantic import BaseModel
from pypdf import PdfReader
from typing import Literal

In [ ]:
from google.colab import userdata
from openai import OpenAI

openai_api_key = userdata.get('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key)

## Extract content from the PDF file

In [ ]:
# IMPROVEMENTS: Extract a function, consider using with PdfReader() in order to close it.
path_to_pdf = Path("/content/The Adventures of Sherlock Holmes.pdf")
pdf_reader = PdfReader(path_to_pdf)

In [ ]:
print(f"There are {pdf_reader.get_num_pages()} pages in total.")

In [ ]:
text_fragments = [page.extract_text() for page in pdf_reader.pages]
text = '\n'.join(text_fragments)

In [ ]:
print(f"There are {len(text)} characters in total.")

## Chunk the PDF content

In [ ]:
# Our chunking implementation will take 1000 characters for each chunk with 200 characters overlap.
# Another idea is to split by '\n' and work with a fixed amount of lines (and overlap).
text_chunks = []

# TODO: Extract a function
start = 0
length = 1000
overlap = 200

while start < len(text):
    end = start + length
    current_chunk = text[start:end]
    text_chunks.append(current_chunk)
    start = end - overlap

## Setup the Vector Database

In [ ]:
chroma_client = PersistentClient("/content/chroma")
chroma_client.delete_collection(name="quotes")
chroma_collection = chroma_client.get_or_create_collection(
    name="quotes",
    embedding_function=OpenAIEmbeddingFunction(api_key=openai_api_key, model_name="text-embedding-3-large")
)

In [ ]:
ids = [f"{i + 1}" for i in range(len(text_chunks))]
documents = [chunk for chunk in text_chunks]

chroma_collection.add(ids=ids, documents=documents)

In [ ]:
# Example:
chroma_collection.query(query_texts="Describe Dr. Watson's role in the adventures.", n_results=5, include=["distances", "documents"])

## Retrieve information

In [ ]:
# First approach (CAG):
# Query related parts of the text
# Extend the system prompt
# Send the input as a user message
# Get response and return its content

# Second approach (2-step RAG):
# Implement tool
# Forced tool calling
# Handle tool calls

# Third approach (RAG):
# Implement tool
# Handle tool calls
def retrieve_information(input: str, strategy: Literal['cag', 'rag'], verbose = True) -> str:
    if strategy == 'cag':
        semantic_search_results = chroma_collection.query(query_texts=input, n_results=5, include=["documents"])
        conversation = [
            { "role": "developer", "content": "You are a knowledgeable assistant. Answer the user's question based ONLY on the provided context from previous interactions. Be specific and concise." },
            { "role": "assistant", "content": f"Here are some quotes that I should refer to later in order to answer the user question:\n\n{'\n=====\n'.join(semantic_search_results["documents"][0])}" },
            { "role": "user", "content": input }
        ]

        if verbose:
            for item in conversation:
                print(f"{item['role']}:")
                print(item['content'])

        response = openai_client.responses.create(
            input=conversation,
            model="gpt-5-nano"
        )

        return response.output_text
    elif strategy == 'rag':
        # TODO: Finish the RAG implementation
        raise RuntimeError("RAG is not supported yet")
    else:
        raise RuntimeError("Invalid strategy")

In [ ]:
retrieve_information("Describe Dr. Watson's role in the adventures.", strategy="cag")

In [ ]:
class IntermediateResponse(BaseModel):
    question: str
    expected_format: Literal['text', 'image', 'speech']


def ask_ai(input: str, verbose = True) -> None:
    intermediate_response = openai_client.responses.parse(
        model="gpt-5-nano",
        input=[
            {
                "role": "developer",
                "content": """
                           Extract the user's core question about a document and the desired response format.
                           The format must be one of: text, image, speech.
                           If no format is mentioned or in case of uncertainty, default to text.
                           If the format is 'image', the question should predispose a creative answer that can be used later to generate an interesting and visually engaging image.
                           """
            },
            { "role": "user", "content": input }
        ],
        text_format=IntermediateResponse
    )

    if verbose:
        print("Intermediate response:")
        print(intermediate_response.output_parsed)

    answer = retrieve_information(intermediate_response.output_parsed.question, strategy="cag", verbose=verbose)

    if intermediate_response.output_parsed.expected_format == 'text':
        print(answer)
    elif intermediate_response.output_parsed.expected_format == 'image':
        print(answer)

        answer_image = openai_client.images.generate(
            model="gpt-image-1-mini",
            prompt=f"""
                   This is the question I asked: \"{input}\".
                   This is the answer I got: \"{answer}\".

                   Create a highly photorealistic image that helps visualize the answer by showing a realistic scene, including the context, environment, and circumstances in which the situation naturally occurs.
                   Generate the image based ONLY on the provided context - do not refer to your built-in knowledge.
                   """,
            size="1024x1024",
            quality="low",
            output_format="png"
        )

        for image in answer_image.data:
            image_bytes = base64.b64decode(image.b64_json)
            display(Image(data=image_bytes, format="png"))
    elif intermediate_response.output_parsed.expected_format == 'speech':
        print(answer)

        answer_speech = openai_client.audio.speech.create(
            input=answer,
            model="gpt-4o-mini-tts",
            voice="alloy"
        )

        path_to_answer = Path("/content/last_answer.mp3")
        answer_speech.write_to_file(path_to_answer)
        display(Audio(path_to_answer))

In [ ]:
ask_ai("What is the name of the story where Sherlock Holmes meets Irene Adler? Give me a text answer.")

In [ ]:
ask_ai("Show me an image of the scene where Holmes disguises himself in 'A Scandal in Bohemia'.")

In [ ]:
ask_ai("Talk to me about the villain in 'The Speckled Band'. I expect the result as speech.")

In [ ]:
ask_ai("Read me the key parts of 'The Five Orange Pips' as audio.")